# Vroegsignalering — Effectiveness Analysis

## Purpose
Early-warning systems for problematic debt are designed to reach households before their situation
escalates. But reaching someone is only the first step — the real question is whether that contact actually
**helps people get back on track**. This notebook looks beyond *"did we make contact?"* to ask
*"did it work?"*, by measuring whether households stay out of trouble after a case is closed.

## What we measure
We define success as a household **not returning with a new signal** in the months after its case is closed.
Around that single idea, the notebook builds up from a broad picture to a focused analysis:

- **The bigger picture** — how many signals come in, who reports them, and how cases are handled and resolved.
- **What actually works** — how the way a household is contacted (the method, the timing, their response)
  relates to whether they return, using statistical models with honest caveats about what the data can and
  cannot prove.

**Privacy first** - Although this notebook works with anonymised data, it never prints or exports individual records; every result is an aggregate statistic or chart.

## The data
Each export shares the same three core tabs — **Signalen** (signals), **Meldingen** (reports/cases) and
**Melders** (reporters) — alongside contact tabs that vary by municipality. The pieces fit together through
shared reference numbers, letting us follow a household's journey from first signal to final outcome.


## 0. Install Dependencies

Run this cell once to install all required Python libraries.
If the packages are already present in your environment this step will be skipped automatically.


In [ ]:
%pip install pandas numpy matplotlib seaborn openpyxl statsmodels scipy --quiet


## 1. Setup

Import all required libraries and configure chart styling.
**Update the `FILE` variable** on the second line of the cell below if the export file has a different name or location.


In [ ]:
# Force a NON-blocking, headless plotting backend. Without this, plt.show() can
# block forever waiting for a GUI window (TkAgg/QtAgg/MacOSX) — which makes the
# first plotting cell appear to "hang". 'inline' renders in the notebook and is
# non-blocking; if IPython is unavailable we fall back to the headless 'Agg'.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    import matplotlib
    matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
FILE     = Path('bi_export_sample.xlsx')   # ← update to real file path if needed
DATE_FMT = '%d-%m-%Y'

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid')
COLORS = sns.color_palette('Set2')

# ══════════════════════════════════════════════════════════════════════════════
#  RESULTS-FILE CAPTURE
#  Every print(), every table shown with display(), and every chart shown with
#  plt.show() is automatically recorded AND written to a single self-contained
#  HTML file (elsa_results_report.html) at the end (Section 15).
#  Send that one file back — it opens in any browser and embeds all charts.
#  Cell outputs still appear in the notebook as normal.
# ══════════════════════════════════════════════════════════════════════════════
import io as _io, base64 as _b64, html as _html, datetime as _dt, builtins as _bi
from IPython.display import display as _ipy_display   # the REAL display (safe to re-import)

REPORT_PATH   = (FILE.parent if hasattr(FILE, 'parent') else Path('.')) / 'elsa_results_report.html'
_REPORT_BLOCKS = []

def _report_add(html_str):
    _REPORT_BLOCKS.append(html_str)

def _report_section(title):
    _report_add(f"<h2 class='sec'>{_html.escape(str(title))}</h2>")

# Tee print() → real stdout + report. builtins.print is always the genuine print,
# so re-running this cell never stacks wrappers on top of each other.
_orig_print = _bi.print
def print(*args, **kwargs):
    _orig_print(*args, **kwargs)
    sep = kwargs.get('sep', ' '); end = kwargs.get('end', '\n')
    _report_add(f"<pre>{_html.escape(sep.join(str(a) for a in args) + end)}</pre>")

# Tee display() → real display + report. _ipy_display is imported straight from
# IPython above, NOT taken from the namespace, so it can never become this wrapper
# (which previously caused infinite recursion when the cell was run twice).
def display(obj, *args, **kwargs):
    _ipy_display(obj, *args, **kwargs)
    try:
        if isinstance(obj, pd.DataFrame):
            _report_add(obj.to_html(border=0))
        elif isinstance(obj, pd.Series):
            _report_add(obj.to_frame().to_html(border=0))
        else:
            _report_add(f"<pre>{_html.escape(str(obj))}</pre>")
    except Exception:
        pass

# Tee plt.show() → embed the current figure as a PNG. Guarded so that re-running
# this cell does not wrap an already-wrapped show() (which would double-capture).
if not getattr(plt.show, '_elsa_wrapped', False):
    _orig_show = plt.show
    def _report_show(*args, **kwargs):
        fig = plt.gcf()
        try:
            buf = _io.BytesIO()
            fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
            buf.seek(0)
            _report_add(f"<img src='data:image/png;base64,{_b64.b64encode(buf.read()).decode()}'/>")
        except Exception:
            pass
        _orig_show(*args, **kwargs)
    _report_show._elsa_wrapped = True
    plt.show = _report_show


### 1a. Load Data

Read all tabs from the Excel file, parse date columns to a consistent format, and define a helper function that converts Dutch `Ja`/`Nee` text values to booleans.
Row counts are printed at the end to confirm the file loaded correctly — no actual data values are shown.


In [ ]:
_report_section('1a. Load Data')
xls = pd.ExcelFile(FILE)

signalen  = pd.read_excel(xls, 'Signalen')
meldingen = pd.read_excel(xls, 'Meldingen')
melders   = pd.read_excel(xls, 'Melders')

FIXED_TABS = {'Signalen', 'Meldingen', 'Melders'}
extra_tabs = {name: pd.read_excel(xls, name)
              for name in xls.sheet_names if name not in FIXED_TABS}

# ── Parse date columns (robust to already-datetime cells and other formats) ───
RAW_DATE_SAMPLES = {}   # original text of date columns, captured BEFORE parsing (for v2)
def parse_dates(df, cols):
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c]
        if pd.api.types.is_datetime64_any_dtype(s):
            continue                      # Excel already gave real date cells
        ex = s.dropna().astype(str).head(3).tolist()   # example raw values reveal the format
        if ex:
            RAW_DATE_SAMPLES.setdefault(c, ex)
        parsed = pd.to_datetime(s, format=DATE_FMT, errors='coerce')
        if parsed.notna().mean() < 0.5:   # fixed format matched almost nothing
            parsed = pd.to_datetime(s, dayfirst=True, errors='coerce')
        df[c] = parsed

parse_dates(signalen,  ['Datum melding', 'Datum doorgezet', 'Startdatum contact'])
parse_dates(meldingen, ['Afgerond d.d.', 'Gezien d.d.'])
for df in extra_tabs.values():
    date_cols = [c for c in df.columns if 'datum' in c.lower() or c.lower() == 'datum']
    parse_dates(df, date_cols)

# ── Ja/Nee → boolean helper ───────────────────────────────────────────────────
def ja_nee(series):
    return series.astype(str).str.strip().str.lower().map({'ja': True, 'nee': False})

# ── Confirm tabs loaded (row counts only — no raw data) ───────────────────────
all_tabs = [('Signalen', signalen), ('Meldingen', meldingen), ('Melders', melders),
            *extra_tabs.items()]
for name, df in all_tabs:
    print(f"  {name:<35} {len(df):>7,} rows")


## 2. Dataset Profile — Knowledge Base for the Next Version

The real export is seen **only once** when this notebook runs, and only this report comes back.
So this section banks a thorough, privacy-safe **profile of the dataset** — schema, exact category labels,
value ranges, date coverage and join integrity — so the next version of the analysis can be written
**without needing to see the raw data again**.

Everything here is aggregate metadata: counts, percentages, ranges and (for low-cardinality, non-identifier
columns) the list of distinct category labels. **No individual records are shown.** A machine-readable JSON
copy of the whole profile is printed at the end of the section (2e) so it can be copied out and reused.


### 2a. Schema — tabs, columns, dtypes, missingness, cardinality

In [ ]:
_report_section('2a. Schema — tabs, columns, dtypes, missingness, cardinality')
# Columns that must never have their raw values shown (identifiers / personal / free text)
SENSITIVE_COLS = {
    'Hash', 'SignaalID', 'Meldingnummer', 'ID', 'Melder ID', 'Postcode',
    'Naam medewerker', 'Melder', 'Melder (submerk)', 'Onderwerp',
    'Toelichting bij de FollowUp', 'Verwijzing anders namelijk', 'Waarnaar is verwezen',
}
CATEGORICAL_MAX = 50   # only list distinct values for columns with at most this many

def schema_table(df):
    return pd.DataFrame({
        'dtype':    df.dtypes.astype(str),
        'non_null': df.count(),
        'null_%':   (df.isnull().sum() / max(len(df), 1) * 100).round(1),
        'n_unique': df.nunique(dropna=True),
    })

for name, df in all_tabs:
    print(f"\n══ {name}: {len(df):,} rows × {df.shape[1]} cols ══")
    display(schema_table(df))


### 2b. Exact Category Labels & Frequencies

For every **low-cardinality, non-identifier** column, the full set of distinct values and their counts.
This is the most valuable part for writing correct mappings/normalisers next time — it pins down the exact
spelling and casing of every contact method, outcome, Ja/Nee token, etc.


In [ ]:
_report_section('2b. Exact Category Labels & Frequencies')
for name, df in all_tabs:
    shown_any = False
    for col in df.columns:
        if col in SENSITIVE_COLS:
            continue
        nu = df[col].nunique(dropna=True)
        if nu < 1 or nu > 1000:
            continue
        if not shown_any:
            print(f"\n══ {name} ══")
            shown_any = True
        if nu <= CATEGORICAL_MAX:
            vc = df[col].value_counts(dropna=False)
            vc.index = vc.index.map(lambda x: '(blank)' if pd.isna(x) else str(x))
            print(f"\n• {col}  ({nu} distinct):")
            print(vc.to_string())
        else:
            # 50 < distinct <= 1000: list only the most common values
            vc = df[col].value_counts().head(15)
            print(f"\n• {col}  ({nu} distinct — showing top 15):")
            print(vc.to_string())


### 2c. Numeric Ranges & Date Coverage

Ranges for numeric columns (to design bands from reality) and coverage / parse-success for date columns.


In [ ]:
_report_section('2c. Numeric Ranges & Date Coverage')
for name, df in all_tabs:
    num = df.select_dtypes(include='number')
    num = num[[c for c in num.columns if c not in SENSITIVE_COLS]]
    if num.shape[1]:
        print(f"\n══ {name} — numeric columns ══")
        display(num.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2))

print("\n══ Date columns — coverage & parse success ══")
date_cols = {'Signalen': ['Datum melding', 'Datum doorgezet', 'Startdatum contact'],
             'Meldingen': ['Afgerond d.d.', 'Gezien d.d.']}
tab_lookup = dict(all_tabs)
for tab, cols in date_cols.items():
    if tab not in tab_lookup:
        continue
    df = tab_lookup[tab]
    for c in cols:
        if c in df.columns and pd.api.types.is_datetime64_any_dtype(df[c]):
            s = df[c]
            ok = s.notna().mean() * 100
            print(f"  {tab}.{c:<20} parsed={ok:5.1f}%  "
                  f"range={s.min().date() if s.notna().any() else 'n/a'} → "
                  f"{s.max().date() if s.notna().any() else 'n/a'}")

print("\n══ Raw date formats (original text, before parsing) ══")
if RAW_DATE_SAMPLES:
    for _col, _ex in RAW_DATE_SAMPLES.items():
        print(f"  {_col:<22} e.g. {_ex}")
else:
    print("  All date columns were already proper date cells (no text parsing needed).")

### 2d. Join Integrity & Structural Distributions

How the tabs link to each other (orphan/duplicate rates) and the structural distributions that determine
what is analysable next time: signals per household, contact attempts per case, follow-up availability,
and cases per municipality.


In [ ]:
_report_section('2d. Join Integrity & Structural Distributions')
print("── Join integrity ──")
sig_with_mld = signalen['Meldingnummer'].notna().mean() * 100
print(f"  Signalen with a Meldingnummer (linkable to Meldingen): {sig_with_mld:.1f}%")

mld_keys = set(meldingen['Meldingnummer'].dropna())
for tab in ['Tussenresultaat', 'Eindresultaat', 'Follow-Up resultaat']:
    if tab in extra_tabs and 'Meldingnummer' in extra_tabs[tab].columns:
        ids = extra_tabs[tab]['Meldingnummer'].dropna()
        match = ids.isin(mld_keys).mean() * 100 if len(ids) else 0
        print(f"  {tab}: {len(ids):,} rows, {match:.1f}% match a Meldingen case")

if 'Melder ID' in signalen.columns:
    melder_ids = set(melders['ID'].dropna())
    m = signalen['Melder ID'].dropna()
    print(f"  Signalen.Melder ID matching Melders registry: "
          f"{m.isin(melder_ids).mean()*100:.1f}%")

dup_mld = int(meldingen['Meldingnummer'].duplicated().sum())
print(f"  Duplicate Meldingnummer in Meldingen: {dup_mld:,}")

print("\n── Recidive structure (signals per household Hash) ──")
sph = signalen.dropna(subset=['Hash']).groupby('Hash').size()
if len(sph):
    print(f"  Households (unique Hash): {len(sph):,}")
    print(f"  Signals per household: mean={sph.mean():.2f}  median={sph.median():.0f}  max={sph.max()}")
    multi = (sph > 1).mean() * 100
    print(f"  Households with >1 signal (recidive present): {multi:.1f}%")

print("\n── Contact attempts per case ──")
att = []
for tab in ['Tussenresultaat', 'Eindresultaat']:
    if tab in extra_tabs:
        att.append(extra_tabs[tab][['Meldingnummer']])
if att:
    apc = pd.concat(att).dropna().groupby('Meldingnummer').size()
    print(f"  Cases with ≥1 attempt: {len(apc):,}")
    print(f"  Attempts per case: mean={apc.mean():.2f}  median={apc.median():.0f}  max={apc.max()}")

print("\n── Follow-up availability for recidive windows ──")
if 'Afgerond d.d.' in meldingen.columns and pd.api.types.is_datetime64_any_dtype(meldingen['Afgerond d.d.']):
    dmax = signalen['Datum melding'].max()
    closed_ct = ja_nee(meldingen['Afgerond'])
    for label, days in {'6 months': 182, '9 months': 273}.items():
        ok = (closed_ct.fillna(False) &
              meldingen['Afgerond d.d.'].notna() &
              ((meldingen['Afgerond d.d.'] + pd.Timedelta(days=days)) <= dmax)).sum()
        print(f"  Closed cases with ≥{label} follow-up: {ok:,}")

print("\n── Cases per municipality (multilevel feasibility) ──")
if 'Gemeente' in meldingen.columns:
    cpm = meldingen['Gemeente'].value_counts()
    print(f"  Municipalities: {cpm.size}  |  cases/municipality "
          f"mean={cpm.mean():.0f} median={cpm.median():.0f} min={cpm.min()} max={cpm.max()}")

print("\n── One household per case? (Meldingnummer -> Hash) ──")
_sl  = signalen.dropna(subset=['Meldingnummer', 'Hash'])
_hpc = _sl.groupby('Meldingnummer')['Hash'].nunique()
_multi = int((_hpc > 1).sum())
print(f"  Cases with linked signals: {len(_hpc):,}")
print(f"  ...mapping to >1 distinct household Hash: {_multi:,} "
      f"({_multi / max(len(_hpc), 1) * 100:.2f}%)")
print("  OK - one household per case; case-level recidive logic holds." if _multi == 0
      else "  WARNING - some cases span multiple households; revisit the one-Hash-per-case assumption.")

print("\n── Signalen -> Meldingen referential integrity ──")
_sig_mld = signalen['Meldingnummer'].dropna()
_orphan  = int((~_sig_mld.isin(mld_keys)).sum())
print(f"  Signals carrying a Meldingnummer: {len(_sig_mld):,}")
print(f"  ...whose Meldingnummer is NOT found in Meldingen (orphans): {_orphan:,} "
      f"({_orphan / max(len(_sig_mld), 1) * 100:.2f}%)")

## 3. Signal Volume & Distribution

How many signals arrive per month, and which signal types are most common?


In [ ]:
_report_section('3. Signal Volume & Distribution')
# Use only valid dates; convert to monthly Timestamps for a clean, fast x-axis.
# (Garbage/NaT dates are dropped so a few bad rows can't blow up the plot range.)
_valid_dates = signalen['Datum melding'].dropna()
monthly = (_valid_dates.dt.to_period('M').dt.to_timestamp()
           .value_counts().sort_index().rename('signals'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(monthly) > 0:
    monthly.plot(ax=axes[0], marker='o', color=COLORS[0], linewidth=2)
axes[0].set_title('Signal Volume per Month')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of signals')

type_counts = signalen['Type melding'].value_counts()
type_counts.plot.barh(ax=axes[1], color=COLORS[1])
axes[1].set_title('Signal Type (Type melding)')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

print(f"Total signals      : {len(signalen):,}")
print(f"Valid signal dates : {len(_valid_dates):,} "
      f"({len(_valid_dates)/max(len(signalen),1)*100:.1f}%)")
if len(_valid_dates) > 0:
    print(f"Date range         : {_valid_dates.min().date()} → {_valid_dates.max().date()}")
else:
    print("Date range         : no parseable dates in 'Datum melding'")
print()
print("Signal type counts:")
print(type_counts.to_string())


### 3a. Reporter (Melder) Analysis

Which organisations are sending signals, and in what volume?

In [ ]:
_report_section('3a. Reporter (Melder) Analysis')
sig_with_melder = signalen.merge(
    melders.rename(columns={'ID': 'Melder ID'}), on='Melder ID', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_reporters = sig_with_melder['Melder'].value_counts().head(15)
top_reporters.sort_values().plot.barh(ax=axes[0], color=COLORS[2])
axes[0].set_title('Top 15 Reporters by Signal Volume')
axes[0].set_xlabel('Number of signals')

type_melder = sig_with_melder['Type melder'].value_counts()
type_melder.plot.pie(ax=axes[1], autopct='%1.1f%%',
                     colors=sns.color_palette('Set2', len(type_melder)),
                     startangle=90)
axes[1].set_title('Reporter Type Distribution (Type melder)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("Reporter type — signal counts:")
print(
    type_melder.rename('signals')
    .to_frame()
    .assign(pct_of_total=(type_melder / len(signalen) * 100).round(1))
    .to_string()
)


## 4. Signal → Report Conversion

A signal is "forwarded" (`Doorgezet = Ja`) when the municipality decides it warrants follow-up.
We measure the overall forwarding rate and whether it differs by signal type or reporter type.


In [ ]:
_report_section('4. Signal → Report Conversion')
signalen['doorgezet_bool'] = ja_nee(signalen['Doorgezet'])
signalen['has_report']     = signalen['Meldingnummer'].notna()

total       = len(signalen)
n_forwarded = int(signalen['doorgezet_bool'].sum())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Overall pie
doorgezet_counts = signalen['Doorgezet'].value_counts(dropna=False)
doorgezet_counts.index = doorgezet_counts.index.fillna('(blank)')
doorgezet_counts.plot.pie(ax=axes[0], autopct='%1.1f%%',
                          colors=[COLORS[1], COLORS[0], COLORS[3]], startangle=90)
axes[0].set_title('Forwarded to Report?\n(Doorgezet)')
axes[0].set_ylabel('')

# By signal type
type_conv = (signalen.groupby('Type melding')['doorgezet_bool']
             .agg(forwarded='sum', total='count'))
type_conv['rate_%'] = type_conv['forwarded'] / type_conv['total'] * 100
type_conv['rate_%'].sort_values().plot.barh(ax=axes[1], color=COLORS[3])
axes[1].axvline(n_forwarded / total * 100, color='red', linestyle='--',
                linewidth=1.5, label=f'Overall {n_forwarded/total*100:.1f}%')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Conversion Rate by Signal Type')
axes[1].set_xlabel('% Forwarded')
axes[1].legend(fontsize=9)

# By reporter type
if 'Type melder' in sig_with_melder.columns:
    sig_with_melder['doorgezet_bool'] = ja_nee(sig_with_melder['Doorgezet'])
    type_melder_conv = (sig_with_melder.groupby('Type melder')['doorgezet_bool']
                        .agg(forwarded='sum', total='count'))
    type_melder_conv['rate_%'] = (type_melder_conv['forwarded']
                                  / type_melder_conv['total'] * 100)
    type_melder_conv['rate_%'].sort_values().plot.barh(ax=axes[2], color=COLORS[4])
    axes[2].axvline(n_forwarded / total * 100, color='red', linestyle='--', linewidth=1.5)
    axes[2].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[2].set_title('Conversion Rate by Reporter Type')
    axes[2].set_xlabel('% Forwarded')

plt.tight_layout()
plt.show()

print(f"Overall forwarding rate: {n_forwarded:,} / {total:,} = {n_forwarded/total*100:.1f}%")
print()
print("Conversion by signal type:")
print(type_conv.sort_values('rate_%', ascending=False)
      .assign(**{'rate_%': type_conv['rate_%'].round(1)})
      .to_string())


## 5. Contact Effectiveness

Once a report exists, the municipality tries to reach the client. We look at:
- Overall reach rate (`Bereikt` in Meldingen)
- Reach rate per contact method and time of day
- How many attempts are typically needed


In [ ]:
_report_section('5. Contact Effectiveness')
meldingen['bereikt_bool'] = ja_nee(meldingen['Bereikt'])
bereikt_rate = meldingen['bereikt_bool'].mean() * 100

contact_dfs = []
for tab_name in ['Tussenresultaat', 'Eindresultaat']:
    if tab_name in extra_tabs:
        df = extra_tabs[tab_name].copy()
        df['_tab'] = tab_name
        contact_dfs.append(df)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bereikt_vc = meldingen['Bereikt'].value_counts(dropna=False)
bereikt_vc.index = bereikt_vc.index.fillna('(blank)')
bereikt_vc.plot.pie(ax=axes[0], autopct='%1.1f%%',
                    colors=[COLORS[1], COLORS[0]], startangle=90)
axes[0].set_title('Client Reached?\n(Meldingen · Bereikt)')
axes[0].set_ylabel('')

if contact_dfs:
    contacts = pd.concat(contact_dfs, ignore_index=True)
    contacts['bereikt_bool'] = ja_nee(contacts['Persoon bereikt'])

    method_reach = (contacts.groupby('Wijze van contactpoging')['bereikt_bool']
                    .agg(reach_rate='mean', n_attempts='count'))
    method_reach['reach_rate'] *= 100
    method_reach['reach_rate'].sort_values().plot.barh(ax=axes[1], color=COLORS[2])
    axes[1].axvline(contacts['bereikt_bool'].mean() * 100, color='red',
                    linestyle='--', linewidth=1.5, label='Overall avg')
    axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[1].set_title('Reach Rate by Contact Method')
    axes[1].set_xlabel('% Person Reached')
    axes[1].legend(fontsize=9)

    dagdeel_reach = (contacts.groupby('Dagdeel')['bereikt_bool']
                     .agg(reach_rate='mean', n_attempts='count'))
    dagdeel_reach['reach_rate'] *= 100
    dagdeel_reach['reach_rate'].sort_values().plot.barh(ax=axes[2], color=COLORS[4])
    axes[2].axvline(contacts['bereikt_bool'].mean() * 100, color='red',
                    linestyle='--', linewidth=1.5)
    axes[2].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[2].set_title('Reach Rate by Time of Day (Dagdeel)')
    axes[2].set_xlabel('% Person Reached')
else:
    for ax in axes[1:]:
        ax.text(0.5, 0.5, 'No contact-attempt tabs\nfound in this export',
                ha='center', va='center', transform=ax.transAxes, fontsize=11)

plt.tight_layout()
plt.show()

print(f"Reach rate (Meldingen · Bereikt): {bereikt_rate:.1f}%")
if contact_dfs:
    print(f"Total contact attempts logged  : {len(contacts):,}")
    print()
    print("Reach rate by contact method:")
    print(method_reach.sort_values('reach_rate', ascending=False)
          .assign(reach_rate=method_reach['reach_rate'].round(1))
          .to_string())


## 6. Client Response & Outcomes

Once the client is reached, do they want help, and what is the final outcome?


In [ ]:
_report_section('6. Client Response & Outcomes')
if 'Eindresultaat' in extra_tabs:
    eind = extra_tabs['Eindresultaat'].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    hulp_vc = eind['Wil de klant hulp'].value_counts(dropna=False)
    hulp_vc.index = hulp_vc.index.fillna('(not recorded)')
    hulp_vc.plot.pie(ax=axes[0], autopct='%1.1f%%',
                     colors=sns.color_palette('Set2', len(hulp_vc)), startangle=90)
    axes[0].set_title('Does the Client Want Help?\n(Wil de klant hulp)')
    axes[0].set_ylabel('')

    result_vc = eind['Wat is het eindresultaat'].value_counts(dropna=False)
    result_vc.index = result_vc.index.fillna('(not recorded)')
    result_vc.sort_values().plot.barh(ax=axes[1], color=COLORS[0])
    axes[1].set_title('Final Outcome (Wat is het eindresultaat)')
    axes[1].set_xlabel('Count')

    plt.tight_layout()
    plt.show()

    print("Outcome breakdown:")
    print(
        result_vc.rename('count')
        .to_frame()
        .assign(pct_=(result_vc / result_vc.sum() * 100).round(1))
        .sort_values('count', ascending=False)
        .to_string()
    )
else:
    print("No 'Eindresultaat' tab found in this export.")


## 7. Case Resolution & Timing

Case status breakdown and time from first signal to case closure.


In [ ]:
_report_section('7. Case Resolution & Timing')
meldingen['afgerond_bool']    = ja_nee(meldingen['Afgerond'])
meldingen['ingetrokken_bool'] = ja_nee(meldingen['Ingetrokken'])
meldingen['onterecht_bool']   = ja_nee(meldingen['Onterecht'])

earliest_signal = (signalen.dropna(subset=['Meldingnummer'])
                   .groupby('Meldingnummer')['Datum melding'].min()
                   .rename('Datum melding'))
mel = meldingen.join(earliest_signal, on='Meldingnummer')
mel['days_to_close'] = (mel['Afgerond d.d.'] - mel['Datum melding']).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_mel = len(meldingen)
status = pd.Series({
    'Resolved\n(Afgerond)':     int(meldingen['afgerond_bool'].sum()),
    'Reached\n(Bereikt)':       int(meldingen['bereikt_bool'].sum()),
    'Withdrawn\n(Ingetrokken)': int(meldingen['ingetrokken_bool'].sum()),
    'Incorrect\n(Onterecht)':   int(meldingen['onterecht_bool'].sum()),
})
bars = axes[0].bar(status.index, status.values, color=COLORS[:4])
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width() / 2, h + 0.3,
                 f'{int(h):,}\n({h/n_mel*100:.1f}%)',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_title(f'Report Status Overview  (n={n_mel:,})')
axes[0].set_ylabel('Count')

dtc = mel['days_to_close'].dropna()
if len(dtc) > 1:
    dtc.hist(ax=axes[1], bins=min(30, len(dtc)), color=COLORS[2], edgecolor='white')
    axes[1].axvline(dtc.median(), color='red', linestyle='--', linewidth=1.5,
                    label=f'Median: {dtc.median():.0f}d')
    axes[1].axvline(dtc.mean(), color='orange', linestyle=':', linewidth=1.5,
                    label=f'Mean: {dtc.mean():.0f}d')
    axes[1].set_title('Days from Signal to Case Closure')
    axes[1].set_xlabel('Days')
    axes[1].set_ylabel('Count')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'Insufficient data\nfor time-to-close',
                ha='center', va='center', transform=axes[1].transAxes, fontsize=11)

plt.tight_layout()
plt.show()

print("Status summary:")
for label, count in status.items():
    print(f"  {label.replace(chr(10), ' '):<30} {count:>6,}  ({count/n_mel*100:.1f}%)")
if len(dtc) > 1:
    print(f"\nTime to close:  median={dtc.median():.0f}d  "
          f"mean={dtc.mean():.0f}d  p90={dtc.quantile(.9):.0f}d")


### 7a. Recidive — Repeated Signals

A recidive signal means the same household has sent a signal before.
This cell shows how the recidive rate evolves month by month and whether households with a recidive history are more or less likely to be forwarded to a report.


In [ ]:
_report_section('7a. Recidive — Repeated Signals')
recidive_cols = [c for c in signalen.columns if c.startswith('Recidive maand')]

if recidive_cols:
    rec_rates = {col.replace('Recidive ', ''): ja_nee(signalen[col]).mean() * 100
                 for col in recidive_cols}
    overall_rec = ja_nee(signalen['Recidive']).mean() * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    pd.Series(rec_rates).plot.bar(ax=axes[0], color=COLORS[0])
    axes[0].axhline(overall_rec, color='red', linestyle='--', linewidth=1.5,
                    label=f'Overall: {overall_rec:.1f}%')
    axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[0].set_title('Recidive Rate by Month Prior')
    axes[0].set_ylabel('% of signals flagged as recidive')
    axes[0].tick_params(axis='x', rotation=0)
    axes[0].legend()

    rec_conv = (signalen
                .groupby(ja_nee(signalen['Recidive']).rename('recidive'))
                ['doorgezet_bool'].mean() * 100)
    rec_conv.index = rec_conv.index.map({True: 'Recidive', False: 'No Recidive'})
    rec_conv.plot.bar(ax=axes[1], color=[COLORS[1], COLORS[2]])
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[1].set_title('Forwarding Rate: Recidive vs No Recidive')
    axes[1].set_ylabel('% Forwarded to Report (Doorgezet)')
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.show()

    print(f"Overall recidive rate: {overall_rec:.1f}%")
    print(pd.Series(rec_rates).round(1).rename('recidive_%').to_string())
else:
    print("No recidive month columns found.")


## 8. End-to-End Effectiveness Funnel

The full journey from signal receipt to resolved case.
Each stage shows the count and the percentage of all incoming signals that reach it.


In [ ]:
_report_section('8. End-to-End Effectiveness Funnel')
n_signals   = len(signalen)
n_forwarded = int(ja_nee(signalen['Doorgezet']).sum())
n_reports   = len(meldingen)
n_reached   = int(meldingen['bereikt_bool'].sum())
n_resolved  = int(meldingen['afgerond_bool'].sum())

stages = ['Signals received',
          'Forwarded to report\n(Doorgezet = Ja)',
          'Reports created',
          'Client reached\n(Bereikt = Ja)']
values = [n_signals, n_forwarded, n_reports, n_reached]

if 'Eindresultaat' in extra_tabs:
    eind = extra_tabs['Eindresultaat']
    hulp_ja = (eind['Wil de klant hulp'].fillna('')
               .str.lower().str.startswith('ja').sum())
    stages.append('Client wants help\n(Wil hulp = Ja)')
    values.append(int(hulp_ja))

stages.append('Case resolved\n(Afgerond = Ja)')
values.append(n_resolved)

palette = sns.color_palette('Blues_r', len(stages))
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(stages)), values, color=palette)
for bar, val in zip(bars, values):
    pct = val / n_signals * 100
    ax.text(bar.get_width() + n_signals * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'{val:,}   ({pct:.1f}% of signals)',
            va='center', fontsize=10)
ax.set_yticks(range(len(stages)))
ax.set_yticklabels(stages, fontsize=11)
ax.set_xlabel('Count')
ax.set_title('End-to-End Effectiveness Funnel', fontsize=14, pad=15)
ax.set_xlim(0, n_signals * 1.45)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nFunnel summary:")
for stage, val in zip(stages, values):
    print(f"  {stage.replace(chr(10), ' '):<48} {val:>6,}  ({val/n_signals*100:.1f}%)")


## 9. Contact Method & 9-Month Outcomes

This is where our distinctive analysis begins. We move beyond *"was contact made?"* to ask whether the
contact had a **lasting effect**.

**Success is defined as: the same household does not generate a new signal after its case is closed.**
Households are tracked using the pseudonymised `Hash` identifier — no personal data is shown. We build a
one-row-per-case table, drawing each case's contact method and the client's help decision from **both
contact tabs** (Tussenresultaat + Eindresultaat).

We report this measure at **two follow-up windows — 6 and 9 months** (Section 9a), with 9 months used as
the primary window for the detailed analysis. We then answer two specific questions:

> **Q1 — Does the contact method influence success?**
> Is there a difference in the signal-reduction rate when contacting via telephone vs door-to-door (or other methods)?

> **Q2 — Is contact alone sufficient when help is refused?**
> When clients decline help, does the contact attempt itself still prevent new signals
> (e.g. because people take action independently or seek help outside the municipality)?

*Note: a case is only included for a given window once enough time has elapsed since closure (e.g. ≥9 months
for the 9-month measure).*

In [ ]:
_report_section('9. Contact Method & 9-Month Outcomes')
# ── Step 1: one row per closed case ──────────────────────────────────────────
meldingen['afgerond_bool'] = ja_nee(meldingen['Afgerond'])
meldingen['bereikt_bool']  = ja_nee(meldingen['Bereikt'])
signalen['doorgezet_bool'] = ja_nee(signalen['Doorgezet'])

# Hash + first signal date per case
signal_per_case = (
    signalen.dropna(subset=['Meldingnummer'])
    .sort_values('Datum melding')
    .groupby('Meldingnummer', as_index=False)
    .first()[['Meldingnummer', 'Hash', 'Datum melding']]
    .rename(columns={'Datum melding': 'first_signal_date'})
)

case_df = meldingen[['Meldingnummer', 'Afgerond d.d.', 'afgerond_bool']].copy()
case_df = case_df.merge(signal_per_case, on='Meldingnummer', how='left')

# ── Contact method & help decision — combine BOTH contact tabs ────────────────
# Tussenresultaat (intermediate) + Eindresultaat (final). Per case we take the
# first attempt where the client was reached; if never reached, the last attempt.
# Combining both tabs keeps cases that only have intermediate contact records.
_contact_src = []
for _t in ['Tussenresultaat', 'Eindresultaat']:
    if _t in extra_tabs:
        _d = extra_tabs[_t].copy()
        _d['_bereikt'] = ja_nee(_d['Persoon bereikt'])
        _keep = ['Meldingnummer', 'Datum contactpoging', 'Wijze van contactpoging',
                 'Wil de klant hulp', '_bereikt']
        _contact_src.append(_d[[c for c in _keep if c in _d.columns]])

if _contact_src:
    _allc = (pd.concat(_contact_src, ignore_index=True)
             .dropna(subset=['Meldingnummer']).sort_values('Datum contactpoging'))
    _reached = (_allc[_allc['_bereikt'] == True]
                .groupby('Meldingnummer', as_index=False).first())
    _nr_ids = set(_allc['Meldingnummer']) - set(_reached['Meldingnummer'])
    _not_reached = (_allc[_allc['Meldingnummer'].isin(_nr_ids)]
                    .groupby('Meldingnummer', as_index=False).last())
    contact_per_case = (pd.concat([_reached, _not_reached], ignore_index=True)
                        .drop(columns=['Datum contactpoging', '_bereikt'], errors='ignore'))
    case_df = case_df.merge(contact_per_case, on='Meldingnummer', how='left')

# ── Step 2: recidive at multiple follow-up windows (6 and 9 months) ──────────
RECIDIVE_WINDOWS = {'6 months': 182, '9 months': 273}
PRIMARY_WINDOW   = '9 months'           # window used by the detailed model & sections
data_max_date    = signalen['Datum melding'].max()

closed = case_df[case_df['afgerond_bool'] == True].copy()

# Recidive — fully vectorised (no per-row apply), so it scales to ~1M rows.
# Join each closed case to its household's signals, then check the day gap.
# Memory ≈ O(rows × avg signals per household); output is a single bool per case.
_sig_events = signalen.dropna(subset=['Hash', 'Datum melding'])[['Hash', 'Datum melding']]
_cl = closed.dropna(subset=['Hash', 'Afgerond d.d.'])[['Meldingnummer', 'Hash', 'Afgerond d.d.']]
_merged = _cl.merge(_sig_events, on='Hash', how='left')
_gap = (_merged['Datum melding'] - _merged['Afgerond d.d.']).dt.days   # days after closure

for _label, _days in RECIDIVE_WINDOWS.items():
    _in_win = (_gap > 0) & (_gap <= _days)
    _flag   = _in_win.groupby(_merged['Meldingnummer']).any()          # one bool per case
    _suff   = (closed['Afgerond d.d.'].notna() &
               ((closed['Afgerond d.d.'] + pd.Timedelta(days=_days)) <= data_max_date))
    _col = f'new_signal_{_days}d'
    closed[_col] = closed['Meldingnummer'].map(_flag).astype('float')  # 1.0/0.0; NaN if no Hash match
    closed[_col] = closed[_col].where(_suff, np.nan)                   # NaN if too recent
    closed[f'sufficient_{_days}d'] = _suff

# Normalised contact-method label (shared by the comparison and later sections)
def _norm_method(v):
    s = str(v).lower()
    if 'telef' in s:                     return 'telefoon'
    if 'huisbez' in s or 'bezoek' in s:  return 'huisbezoek'
    if 'sms' in s or 'whats' in s:       return 'sms/whatsapp'
    if 'mail' in s:                      return 'email'
    if 'brief' in s or 'schrift' in s:   return 'brief'
    return 'anders/onbekend'
if 'Wijze van contactpoging' in closed.columns:
    closed['method'] = closed['Wijze van contactpoging'].apply(_norm_method)

# Primary analysable set (enough elapsed time for the primary window)
_pdays = RECIDIVE_WINDOWS[PRIMARY_WINDOW]
analysable = closed[closed[f'sufficient_{_pdays}d']].copy()
analysable['new_signal_9m'] = analysable[f'new_signal_{_pdays}d']   # alias for downstream
n_analysable = len(analysable)

print(f"Closed cases with >= {PRIMARY_WINDOW} follow-up available: {n_analysable:,}")
print(f"  (Excluded: not yet closed, or closed too recently for the {PRIMARY_WINDOW} window)")
if n_analysable > 0:
    overall_new_signal_rate = analysable['new_signal_9m'].mean() * 100
    overall_success_rate    = 100 - overall_new_signal_rate
    print(f"\nOverall success rate (no new signal, {PRIMARY_WINDOW}): {overall_success_rate:.1f}%")
    print(f"Overall new-signal rate ({PRIMARY_WINDOW})              : {overall_new_signal_rate:.1f}%")
else:
    print("\nInsufficient follow-up data — analyses below require more time coverage.")


### 9a. Recidive at 6 vs 9 Months

The same success measure (no new signal after closure) at two follow-up horizons.
The analysable set differs per window — the 9-month figure needs more elapsed time, so it covers fewer cases.


In [ ]:
_report_section('9a. Recidive at 6 vs 9 Months')
rows = []
for _label, _days in RECIDIVE_WINDOWS.items():
    col, sc = f'new_signal_{_days}d', f'sufficient_{_days}d'
    sub = closed[closed[sc] & closed[col].notna()]
    if len(sub):
        succ = (1 - sub[col].mean()) * 100
        rows.append({'window': _label, 'n_cases': len(sub),
                     'success_rate_pct': round(succ, 1),
                     'new_signal_pct': round(100 - succ, 1)})

if not rows:
    print("Not enough follow-up time to report either window.")
else:
    cmp = pd.DataFrame(rows)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    axes[0].bar(cmp['window'], cmp['success_rate_pct'], color=[COLORS[0], COLORS[1]])
    axes[0].set_title('Overall Success Rate (no new signal)')
    axes[0].set_ylabel('% of cases')
    axes[0].set_ylim(0, 100)
    for i, r in cmp.iterrows():
        axes[0].text(i, r['success_rate_pct'] + 1,
                     f"{r['success_rate_pct']:.1f}%\n(n={int(r['n_cases']):,})",
                     ha='center', fontsize=9)

    # success by contact method, per window (grouped)
    if 'method' in closed.columns:
        meth_rows = []
        for _label, _days in RECIDIVE_WINDOWS.items():
            col, sc = f'new_signal_{_days}d', f'sufficient_{_days}d'
            sub = closed[closed[sc] & closed[col].notna()]
            g = (1 - sub.groupby('method')[col].mean()) * 100
            for m, v in g.items():
                meth_rows.append({'method': m, 'window': _label, 'success_rate_pct': v})
        mtab = (pd.DataFrame(meth_rows)
                .pivot(index='method', columns='window', values='success_rate_pct'))
        mtab.plot.bar(ax=axes[1], color=[COLORS[0], COLORS[1]])
        axes[1].set_title('Success Rate by Contact Method')
        axes[1].set_ylabel('% no new signal')
        axes[1].set_xlabel('')
        axes[1].tick_params(axis='x', rotation=20)
        axes[1].legend(title='window')

    plt.tight_layout()
    plt.show()
    print("6-month vs 9-month recidive success:")
    print(cmp.to_string(index=False))


### Q1 — Contact Method vs 9-Month Success

Does contacting via telephone vs door-to-door (or other methods) produce different success rates?


In [ ]:
_report_section('Q1 — Contact Method vs 9-Month Success')
if n_analysable == 0:
    print("Skipped — no cases with sufficient follow-up.")
elif 'Wijze van contactpoging' not in analysable.columns:
    print("Skipped — 'Wijze van contactpoging' column not found (Eindresultaat tab missing?).")
else:
    q1 = (
        analysable.dropna(subset=['new_signal_9m', 'Wijze van contactpoging'])
        .groupby('Wijze van contactpoging')['new_signal_9m']
        .agg(n_cases='count', new_signal_rate='mean')
        .assign(success_rate_pct=lambda d: (1 - d['new_signal_rate']) * 100,
                new_signal_pct =lambda d: d['new_signal_rate'] * 100)
        .sort_values('success_rate_pct')
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Success rate by method
    bars = axes[0].barh(q1.index, q1['success_rate_pct'], color=COLORS[1])
    axes[0].axvline(overall_success_rate, color='red', linestyle='--', linewidth=1.5,
                    label=f'Overall: {overall_success_rate:.1f}%')
    axes[0].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[0].set_title('9-Month Success Rate by Contact Method\n(no new signal within 9 months)')
    axes[0].set_xlabel('% of cases with no new signal')
    axes[0].legend(fontsize=9)
    for bar, (_, row) in zip(bars, q1.iterrows()):
        axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                     f'{row["success_rate_pct"]:.1f}%  (n={int(row["n_cases"])})',
                     va='center', fontsize=9)
    axes[0].set_xlim(0, 115)

    # Stacked: success vs new signal
    q1_plot = q1[['success_rate_pct', 'new_signal_pct']].rename(
        columns={'success_rate_pct': 'No new signal (success)',
                 'new_signal_pct':   'New signal within 9m'})
    q1_plot.plot.barh(ax=axes[1], stacked=True,
                      color=[COLORS[1], COLORS[3]])
    axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[1].set_title('Success vs New Signal — by Contact Method')
    axes[1].set_xlabel('% of cases')
    axes[1].legend(loc='lower right', fontsize=9)

    plt.suptitle('Q1: Does contact method influence 9-month signal reduction?',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    print("\nQ1 — Contact method outcome summary:")
    print(q1[['n_cases', 'success_rate_pct', 'new_signal_pct']].round(1).to_string())


### Q2 — Is Contact Alone Sufficient When Help Is Refused?

When clients decline help, does receiving a contact attempt still reduce subsequent signals?
A high no-new-signal rate among refusers would suggest that contact prompts people to
seek help independently or resolve the situation themselves.


In [ ]:
_report_section('Q2 — Is Contact Alone Sufficient When Help Is Refused?')
if n_analysable == 0:
    print("Skipped — no cases with sufficient follow-up.")
elif 'Wil de klant hulp' not in analysable.columns:
    print("Skipped — 'Wil de klant hulp' column not found (Eindresultaat tab missing?).")
else:
    def classify_help(val):
        if pd.isna(val) or str(val).strip() == '':
            return np.nan
        v = str(val).lower().strip()
        if v.startswith('ja'):
            return 'Accepted help'
        elif v.startswith('nee'):
            return 'Refused help'
        return 'Other / Unknown'

    analysable['help_decision'] = analysable['Wil de klant hulp'].apply(classify_help)

    q2 = (
        analysable.dropna(subset=['new_signal_9m', 'help_decision'])
        .groupby('help_decision')['new_signal_9m']
        .agg(n_cases='count', new_signal_rate='mean')
        .assign(success_rate_pct=lambda d: (1 - d['new_signal_rate']) * 100,
                new_signal_pct =lambda d: d['new_signal_rate'] * 100)
    )

    help_order  = ['Accepted help', 'Refused help', 'Other / Unknown']
    q2 = q2.reindex([h for h in help_order if h in q2.index])
    bar_colors  = [COLORS[1], COLORS[3], COLORS[0]][:len(q2)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    bars = axes[0].bar(q2.index, q2['success_rate_pct'], color=bar_colors)
    axes[0].axhline(overall_success_rate, color='red', linestyle='--', linewidth=1.5,
                    label=f'Overall: {overall_success_rate:.1f}%')
    axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[0].set_title('9-Month Success Rate by Help Decision')
    axes[0].set_ylabel('% with no new signal within 9 months')
    axes[0].tick_params(axis='x', rotation=15)
    axes[0].legend(fontsize=9)
    for bar, (_, row) in zip(bars, q2.iterrows()):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                     f'{row["success_rate_pct"]:.1f}%\n(n={int(row["n_cases"])})',
                     ha='center', va='bottom', fontsize=9)
    axes[0].set_ylim(0, 115)

    # Stacked breakdown
    q2_plot = q2[['success_rate_pct', 'new_signal_pct']].rename(
        columns={'success_rate_pct': 'No new signal (success)',
                 'new_signal_pct':   'New signal within 9m'})
    q2_plot.plot.bar(ax=axes[1], stacked=True, color=[COLORS[1], COLORS[3]])
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[1].set_title('Success vs New Signal — by Help Decision')
    axes[1].set_ylabel('% of cases')
    axes[1].tick_params(axis='x', rotation=15)
    axes[1].legend(loc='lower right', fontsize=9)

    plt.suptitle('Q2: Does contact alone prevent new signals when help is refused?',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    print("\nQ2 — Help decision outcome summary:")
    print(q2[['n_cases', 'success_rate_pct', 'new_signal_pct']].round(1).to_string())

    if 'Refused help' in q2.index:
        refused_success = q2.loc['Refused help', 'success_rate_pct']
        accepted_success = q2.loc['Accepted help', 'success_rate_pct'] \
                           if 'Accepted help' in q2.index else None
        print(f"\n── Key finding ──")
        print(f"Refusers with no new signal within 9 months : {refused_success:.1f}%")
        if accepted_success is not None:
            diff = refused_success - accepted_success
            direction = 'higher' if diff > 0 else 'lower'
            print(f"Accepters with no new signal within 9 months: {accepted_success:.1f}%")
            print(f"Difference (refusers vs accepters)          : {abs(diff):.1f}pp {direction}")
        if refused_success >= 50:
            print("→ Majority of refusers did NOT generate new signals.")
            print("  Contact alone may be sufficient for these households.")
        else:
            print("→ Majority of refusers DID generate new signals.")
            print("  Contact alone appears insufficient — continued follow-up may be needed.")


### Q1 × Q2 — Contact Method and Help Decision Combined

This chart combines both dimensions: it shows the 9-month success rate split by contact method *and* whether the client accepted help.
Groups with fewer than 5 cases are excluded to avoid drawing conclusions from small samples.


In [ ]:
_report_section('Q1 × Q2 — Contact Method and Help Decision Combined')
if n_analysable > 0 and all(
    c in analysable.columns for c in ['Wijze van contactpoging', 'help_decision']
):
    combined = (
        analysable.dropna(subset=['new_signal_9m', 'Wijze van contactpoging', 'help_decision'])
        .groupby(['Wijze van contactpoging', 'help_decision'])['new_signal_9m']
        .agg(n_cases='count', new_signal_rate='mean')
        .assign(success_rate_pct=lambda d: (1 - d['new_signal_rate']) * 100)
    )

    # Only show combinations with enough cases to be meaningful (n >= 5)
    combined = combined[combined['n_cases'] >= 5]

    if combined.empty:
        print("Insufficient cases per combination (need n ≥ 5 per group).")
    else:
        pivot = combined['success_rate_pct'].unstack('help_decision').fillna(np.nan)

        fig, ax = plt.subplots(figsize=(13, 5))
        pivot.plot.bar(ax=ax, color=COLORS[:len(pivot.columns)])
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.axhline(overall_success_rate, color='red', linestyle='--', linewidth=1.5,
                   label=f'Overall: {overall_success_rate:.1f}%')
        ax.set_title('9-Month Success Rate: Contact Method × Help Decision\n'
                     '(no new signal within 9 months; groups with n < 5 excluded)',
                     fontsize=12)
        ax.set_ylabel('% with no new signal')
        ax.set_xlabel('Contact method')
        ax.tick_params(axis='x', rotation=20)
        ax.legend(title='Help decision', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
        plt.tight_layout()
        plt.show()

        print("\nCombined summary (n_cases  |  success_rate_%):")
        summary = combined[['n_cases', 'success_rate_pct']].round(1)
        print(summary.to_string())
elif n_analysable > 0:
    print("Skipped — requires both Eindresultaat tab and sufficient follow-up data.")


## 10. Effectiveness Model — What Reduces Recidive?

This is the core stakeholder question: **does the intervention actually prevent households from coming back?**

We estimate a **logistic regression** where the outcome is **success = no new signal within 9 months** of case closure.
Coefficients are reported as **odds ratios** (same convention as Divosa): OR > 1 = higher chance of success than
the reference group; OR < 1 = lower chance. To respect the nested structure of the data (cases clustered within
municipalities), we use **cluster-robust standard errors** on `Gemeente` — a pragmatic, scalable stand-in for the
full multilevel model Divosa used on the *reach* outcome.

> **Important caveats (observational data, not a randomized experiment)**
> - Caseworkers *choose* who gets which contact method, so estimates are **associational, not causal**.
>   A method may look effective simply because it is used on easier cases.
> - The model controls for the confounders available in the export (whether reached, contact method, age,
>   debt amount, signal type), but unobserved factors may remain. *Help decision* is intentionally left out of
>   this model (it is a post-reach mediator) and is analysed separately in Section 9.
> - Read odds ratios *relative to the reference group*; compare two groups only if their confidence intervals do not overlap.


In [ ]:
_report_section('10. Effectiveness Model — What Reduces Recidive?')
import statsmodels.formula.api as smf
import statsmodels.api as sm

# ── Build the case-level modelling table (reuse `analysable` from Section 9) ──
if n_analysable == 0 or 'new_signal_9m' not in analysable.columns:
    model_df = None
    print("Effectiveness model skipped — no cases with ≥9 months follow-up.")
else:
    # Enrich each closed case with covariates from its first linked signal
    cov_cols = ['Meldingnummer', 'Leeftijd', 'Achterstandsbedrag', 'Termijnbedrag',
                'Gemeente', 'Wijk', 'Type melding', 'Soort melding', 'Minderjarige kinderen']
    first_sig = (signalen.dropna(subset=['Meldingnummer'])
                 .sort_values('Datum melding')
                 .groupby('Meldingnummer', as_index=False).first())
    first_sig = first_sig[[c for c in cov_cols if c in first_sig.columns]]

    model_df = analysable.merge(first_sig, on='Meldingnummer', how='left')
    model_df = model_df.merge(meldingen[['Meldingnummer', 'bereikt_bool']],
                              on='Meldingnummer', how='left')

    # ── Derived features ──────────────────────────────────────────────────────
    model_df['success']  = (~model_df['new_signal_9m'].astype('boolean')).astype('float')
    model_df['reached']  = model_df['bereikt_bool'].map({True: 'reached', False: 'not_reached'})

    # Gemeente as clustering group (fill blanks so clustering never fails)
    model_df['Gemeente'] = model_df['Gemeente'].fillna('Onbekend').astype(str) \
        if 'Gemeente' in model_df.columns else 'Onbekend'

    # Age bands (handle numeric age or pre-banded text)
    if 'Leeftijd' in model_df.columns and pd.api.types.is_numeric_dtype(model_df['Leeftijd']):
        model_df['age_band'] = pd.cut(model_df['Leeftijd'],
                                      bins=[0, 25, 45, 65, 200],
                                      labels=['18-25', '26-45', '46-65', '66+'])
    elif 'Leeftijd' in model_df.columns:
        model_df['age_band'] = model_df['Leeftijd'].astype(str)

    # Debt amount bands (Divosa-style)
    if 'Achterstandsbedrag' in model_df.columns and pd.api.types.is_numeric_dtype(model_df['Achterstandsbedrag']):
        model_df['debt_band'] = pd.cut(model_df['Achterstandsbedrag'],
                                       bins=[-1, 250, 1000, 5000, 1e12],
                                       labels=['0-250', '250-1k', '1k-5k', '5k+'])

    # Normalised contact method label
    def norm_method(v):
        s = str(v).lower()
        if 'telef' in s:                     return 'telefoon'
        if 'huisbez' in s or 'bezoek' in s:  return 'huisbezoek'
        if 'sms' in s or 'whats' in s:       return 'sms/whatsapp'
        if 'mail' in s:                      return 'email'
        if 'brief' in s or 'schrift' in s:   return 'brief'
        return 'anders/onbekend'
    if 'Wijze van contactpoging' in model_df.columns:
        model_df['method'] = model_df['Wijze van contactpoging'].apply(norm_method)

    n_success = int(model_df['success'].sum())
    print(f"Modelling table: {len(model_df):,} closed cases with ≥9 months follow-up")
    print(f"  Success (no new signal within 9m): {n_success:,} ({n_success/len(model_df)*100:.1f}%)")


In [ ]:
# ── Fit the logistic regression with cluster-robust SEs on municipality ───────
def odds_ratio_table(res):
    ci = res.conf_int()
    return pd.DataFrame({
        'odds_ratio': np.exp(res.params),
        'CI_low':     np.exp(ci[0]),
        'CI_high':    np.exp(ci[1]),
        'p_value':    res.pvalues,
    }).drop(index='Intercept', errors='ignore')

if model_df is None or model_df['success'].nunique() < 2:
    or_tab = None
    print("Regression skipped — outcome has no variation or no analysable cases.")
else:
    # Predictors. NOTE: 'help_decision' is deliberately excluded — it is only
    # recorded when a client is reached, so it is a post-reach *mediator*, not a
    # confounder. Including it would collapse the 'reached' contrast (all
    # complete cases would be 'reached'). Help decision is analysed descriptively
    # in Section 9 (Q2) instead.
    pref_ref = {'reached': 'not_reached', 'method': 'brief',
                'age_band': '18-25', 'debt_band': '0-250'}
    predictors = ['reached', 'method', 'age_band', 'debt_band', 'Type melding']

    # Build the complete-case frame FIRST, then derive terms from levels actually present
    fit_cols = ['success', 'Gemeente'] + [c for c in predictors if c in model_df.columns]
    d = model_df[fit_cols].copy()
    for c in fit_cols:
        if c not in ('success',):
            d[c] = d[c].astype('object')
    d = d.dropna()

    def cat_term(col):
        levels = set(d[col].astype(str))
        if len(levels) < 2:
            return None                      # no contrast possible
        ref = pref_ref.get(col)
        if ref is not None and ref in levels:
            return f"C(Q('{col}'), Treatment('{ref}'))"
        return f"C(Q('{col}'))"

    terms = [t for t in (cat_term(c) for c in predictors if c in d.columns) if t]

    if not terms or d['success'].nunique() < 2 or len(d) < 50:
        or_tab = None
        print(f"Regression skipped — only {len(d):,} usable complete cases "
              f"or no predictor variation in this export.")
    else:
        formula = "success ~ " + " + ".join(terms)
        print(f"Formula: {formula}")
        print(f"Rows used (complete cases): {len(d):,}\n")
        try:
            res = smf.logit(formula, data=d).fit(
                disp=0, cov_type='cluster', cov_kwds={'groups': d['Gemeente']})
            or_tab = odds_ratio_table(res)
            sig = or_tab[or_tab['p_value'] < 0.05]
            print(f"Model converged. {len(or_tab)} coefficients, "
                  f"{len(sig)} significant at p<0.05.")
            display(or_tab.round(3))
        except Exception as e:
            or_tab = None
            print(f"Model did not converge / failed: {type(e).__name__}: {e}")
            print("On the real 1M-record export this typically succeeds; "
                  "on tiny samples it cannot estimate.")


In [ ]:
# ── Forest plot of odds ratios (recidive-reduction effects) ───────────────────
if or_tab is not None and len(or_tab) > 0:
    t = or_tab.sort_values('odds_ratio')
    fig, ax = plt.subplots(figsize=(11, max(4, 0.45 * len(t))))
    y = np.arange(len(t))
    xerr = np.vstack([t['odds_ratio'] - t['CI_low'], t['CI_high'] - t['odds_ratio']])
    colors = ['#2ca02c' if lo > 1 else '#d62728' if hi < 1 else '#7f7f7f'
              for lo, hi in zip(t['CI_low'], t['CI_high'])]
    ax.errorbar(t['odds_ratio'], y, xerr=xerr, fmt='o', color='black',
                ecolor='gray', capsize=3, markersize=5, linewidth=0)
    for yi, (orv, c) in enumerate(zip(t['odds_ratio'], colors)):
        ax.plot(orv, yi, 'o', color=c, markersize=8)
    ax.axvline(1.0, color='red', linestyle='--', linewidth=1.5)
    ax.set_yticks(y)
    def clean_label(s):
        return (s.replace("C(Q('", '').replace("')", '').replace('C(', '')
                 .replace(', Treatment(', ' vs ref ').replace('))', ')')
                 .replace("'", '').replace('[T.', ' = ').replace(']', ''))
    ax.set_yticklabels([clean_label(s) for s in t.index], fontsize=9)
    ax.set_xlabel('Odds ratio for SUCCESS (no new signal within 9 months)')
    ax.set_title('Effectiveness Model — drivers of recidive reduction\n'
                 '(green = significantly protective, red = significantly worse, grey = n.s.)',
                 fontsize=12)
    ax.set_xscale('log')
    plt.tight_layout()
    plt.show()
    print("OR > 1  → higher chance of NO new signal (better, more effective)")
    print("OR < 1  → lower chance of NO new signal (worse)")
else:
    print("No odds-ratio table to plot.")


## 11. Targeting Playbook — Which Method, For Whom?

Average effects hide important variation. The same method can work well for one group and poorly for another.
Here we break the **recidive-reduction success rate** down by **contact method × household profile** (age, debt size),
then derive a simple recommendation: *for each profile, which method best prevents a return?*

This mirrors Divosa's subgroup approach (age × method) — but on the **recidive** outcome, not the reach outcome,
so it answers a different question: not "who do we reach?" but "**for whom does contact actually work?**"

> Cells with fewer than 30 cases are hidden to avoid conclusions from small samples.


In [ ]:
_report_section('11. Targeting Playbook — Which Method, For Whom?')
MIN_CELL = 30   # minimum cases per method×profile cell to report

def targeting_matrix(df, profile_col, title):
    if df is None or profile_col not in df.columns or 'method' not in df.columns:
        print(f"Targeting by {profile_col}: skipped (column unavailable).")
        return
    g = (df.dropna(subset=['success', profile_col, 'method'])
         .groupby([profile_col, 'method'])
         .agg(n=('success', 'size'), rate=('success', 'mean')))
    g = g[g['n'] >= MIN_CELL]
    if g.empty:
        print(f"Targeting by {profile_col}: no cells with ≥{MIN_CELL} cases.")
        return

    rate_pivot = (g['rate'] * 100).unstack('method')
    n_pivot    = g['n'].unstack('method')

    fig, ax = plt.subplots(figsize=(1.6 * rate_pivot.shape[1] + 3, 0.7 * len(rate_pivot) + 2))
    sns.heatmap(rate_pivot, annot=True, fmt='.0f', cmap='Greens', cbar_kws={'label': '% success'},
                linewidths=0.5, ax=ax)
    ax.set_title(f'{title}\n9-month success rate (%) — no new signal', fontsize=12)
    ax.set_xlabel('Contact method')
    ax.set_ylabel(profile_col)
    plt.tight_layout()
    plt.show()

    # Best method per profile (among sufficiently-sized cells)
    best = rate_pivot.idxmax(axis=1)
    best_val = rate_pivot.max(axis=1)
    rec = pd.DataFrame({'recommended_method': best,
                        'success_rate_pct': best_val.round(1)})
    print(f"Recommended method per {profile_col} (highest recidive reduction, n≥{MIN_CELL}):")
    print(rec.to_string())
    return rec

print("=== Targeting by age band ===")
targeting_matrix(model_df, 'age_band', 'Method effectiveness by age')
print("\n=== Targeting by debt size ===")
targeting_matrix(model_df, 'debt_band', 'Method effectiveness by debt size')


## 12. Escalation Sequences & Speed of Contact

Two operational levers that Divosa did **not** examine (they looked only at the *count* of attempts → reach):

1. **Escalation sequence** — does the *order* of methods matter? e.g. does `sms → telefoon` prevent return
   better than `telefoon → huisbezoek`?
2. **Speed of contact** — does reaching a household *faster* after the first signal reduce recidive?

Both are measured here on the **recidive** outcome (no new signal within 9 months).


In [ ]:
_report_section('12. Escalation Sequences & Speed of Contact')
# ── Rebuild all contact attempts with their dates (self-contained) ────────────
seq_dfs = []
for tab_name in ['Tussenresultaat', 'Eindresultaat']:
    if tab_name in extra_tabs:
        t = extra_tabs[tab_name][['Meldingnummer', 'Datum contactpoging', 'Wijze van contactpoging']].copy()
        seq_dfs.append(t)

if not seq_dfs or model_df is None or 'new_signal_9m' not in analysable.columns:
    print("Escalation/speed analysis skipped — needs contact tabs and follow-up data.")
else:
    attempts = pd.concat(seq_dfs, ignore_index=True).dropna(subset=['Meldingnummer'])

    def norm_method2(v):
        s = str(v).lower()
        if 'telef' in s:                     return 'telefoon'
        if 'huisbez' in s or 'bezoek' in s:  return 'huisbezoek'
        if 'sms' in s or 'whats' in s:       return 'sms/whatsapp'
        if 'mail' in s:                      return 'email'
        if 'brief' in s or 'schrift' in s:   return 'brief'
        return 'anders'
    attempts['m'] = attempts['Wijze van contactpoging'].apply(norm_method2)
    attempts = attempts.sort_values(['Meldingnummer', 'Datum contactpoging'])

    # First two methods per case → sequence label
    seq = (attempts.groupby('Meldingnummer')['m']
           .apply(lambda s: ' → '.join(list(s)[:2]))
           .rename('sequence').reset_index())

    outcome = analysable[['Meldingnummer', 'new_signal_9m']].copy()
    outcome['success'] = (~outcome['new_signal_9m'].astype('boolean')).astype('float')
    seq = seq.merge(outcome, on='Meldingnummer', how='inner').dropna(subset=['success'])

    seq_stats = (seq.groupby('sequence')
                 .agg(n=('success', 'size'), success_rate=('success', 'mean')))
    seq_stats = seq_stats[seq_stats['n'] >= 30].sort_values('success_rate', ascending=False)

    if seq_stats.empty:
        print("No contact sequence has ≥30 cases with follow-up.")
    else:
        fig, ax = plt.subplots(figsize=(11, max(4, 0.5 * len(seq_stats))))
        (seq_stats['success_rate'] * 100).sort_values().plot.barh(ax=ax, color=COLORS[2])
        ax.set_title('Recidive Reduction by Contact Sequence (first two methods)\n'
                     'success = no new signal within 9 months; sequences with n≥30', fontsize=12)
        ax.set_xlabel('% success')
        for i, (v, n) in enumerate(zip(seq_stats['success_rate'].sort_values() * 100,
                                       seq_stats['n'].loc[seq_stats['success_rate'].sort_values().index])):
            ax.text(v + 0.3, i, f'{v:.0f}% (n={int(n)})', va='center', fontsize=8)
        plt.tight_layout()
        plt.show()
        print("Sequence success rates:")
        print((seq_stats.assign(success_rate=(seq_stats['success_rate']*100).round(1))).to_string())


In [ ]:
# ── Speed of contact: days from first signal to first contact attempt ─────────
if not seq_dfs or model_df is None or 'new_signal_9m' not in analysable.columns:
    print("Speed analysis skipped — needs contact tabs and follow-up data.")
else:
    first_contact = (attempts.groupby('Meldingnummer')['Datum contactpoging']
                     .min().rename('first_contact_date').reset_index())
    first_signal = (signalen.dropna(subset=['Meldingnummer'])
                    .groupby('Meldingnummer')['Datum melding']
                    .min().rename('first_signal_date').reset_index())

    speed = (analysable[['Meldingnummer', 'new_signal_9m']]
             .merge(first_contact, on='Meldingnummer', how='left')
             .merge(first_signal, on='Meldingnummer', how='left'))
    speed['days_to_contact'] = (speed['first_contact_date'] - speed['first_signal_date']).dt.days
    speed['success'] = (~speed['new_signal_9m'].astype('boolean')).astype('float')
    speed = speed.dropna(subset=['days_to_contact', 'success'])
    speed = speed[speed['days_to_contact'].between(0, 365)]   # drop impossible/extreme values

    if len(speed) < 30:
        print("Not enough cases with valid contact timing for a speed analysis.")
    else:
        speed['speed_band'] = pd.cut(speed['days_to_contact'],
                                     bins=[-1, 7, 14, 30, 60, 365],
                                     labels=['0-7d', '8-14d', '15-30d', '31-60d', '60d+'])
        sb = (speed.groupby('speed_band')
              .agg(n=('success', 'size'), success_rate=('success', 'mean')))
        sb = sb[sb['n'] >= 30]

        fig, ax = plt.subplots(figsize=(10, 5))
        (sb['success_rate'] * 100).plot.bar(ax=ax, color=COLORS[0])
        ax.set_title('Does Faster Contact Reduce Recidive?\n'
                     'success = no new signal within 9 months', fontsize=12)
        ax.set_ylabel('% success')
        ax.set_xlabel('Days from first signal to first contact attempt')
        ax.tick_params(axis='x', rotation=0)
        for i, (v, n) in enumerate(zip(sb['success_rate'] * 100, sb['n'])):
            ax.text(i, v + 0.4, f'{v:.0f}%\n(n={int(n)})', ha='center', fontsize=8)
        plt.tight_layout()
        plt.show()
        print(f"Median days to first contact: {speed['days_to_contact'].median():.0f}")
        print((sb.assign(success_rate=(sb['success_rate']*100).round(1))).to_string())


## 13. Does Follow-Up Contact Reduce Recidive?

The `Follow-Up resultaat` tab records *continued* contact after the main case handling.
Here we test whether closed cases that received a follow-up contact have a **lower recidive rate**
(no new signal within the primary window) than cases without one.

> **Caveat:** follow-up is not assigned at random — caseworkers may schedule it precisely for the
> cases they judge most at risk. A higher recidive rate among followed-up cases could therefore reflect
> *who* gets follow-up, not its effect. Read this as associational.


In [ ]:
_report_section('13. Does Follow-Up Contact Reduce Recidive?')
if 'Follow-Up resultaat' not in extra_tabs or n_analysable == 0:
    print("Follow-up analysis skipped — needs the 'Follow-Up resultaat' tab and follow-up data.")
else:
    fu = extra_tabs['Follow-Up resultaat']
    fu_ids = set(fu['Meldingnummer'].dropna()) if 'Meldingnummer' in fu.columns else set()

    a = analysable.dropna(subset=['new_signal_9m']).copy()
    a['has_followup'] = a['Meldingnummer'].isin(fu_ids)

    grp = (a.groupby('has_followup')['new_signal_9m']
           .agg(n_cases='size', new_signal_rate='mean'))
    grp['success_rate_pct'] = (1 - grp['new_signal_rate']) * 100
    grp.index = grp.index.map({True: 'With follow-up', False: 'No follow-up'})

    if grp['n_cases'].min() < 30 or len(grp) < 2:
        print("Not enough cases in each group (need ≥30 with and without follow-up).")
        print(grp.round(1).to_string())
    else:
        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(grp.index, grp['success_rate_pct'], color=[COLORS[2], COLORS[1]])
        ax.axhline(overall_success_rate, color='red', linestyle='--', linewidth=1.5,
                   label=f'Overall: {overall_success_rate:.1f}%')
        ax.set_title('Recidive Success Rate: Follow-Up vs No Follow-Up\n'
                     '(success = no new signal within the primary window)', fontsize=12)
        ax.set_ylabel('% no new signal')
        ax.set_ylim(0, 100)
        ax.legend(fontsize=9)
        for bar, (_, r) in zip(bars, grp.iterrows()):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f"{r['success_rate_pct']:.1f}%\n(n={int(r['n_cases']):,})",
                    ha='center', fontsize=9)
        plt.tight_layout()
        plt.show()

        diff = (grp.loc['With follow-up', 'success_rate_pct']
                - grp.loc['No follow-up', 'success_rate_pct'])
        print("Follow-up vs recidive success:")
        print(grp[['n_cases', 'success_rate_pct']].round(1).to_string())
        print(f"\nDifference (with − without follow-up): {diff:+.1f} percentage points")


## 14. Generate the Results File to Send Back

Run this final cell **after running all the cells above**. It writes a single self-contained file —
**`elsa_results_report.html`** — into the same folder as the data, containing every table, chart and
printed summary produced above (charts are embedded directly in the file).

**This is the only file you need to send back.** It opens in any web browser, and to convert it to PDF you
can simply open it and use the browser's *Print → Save as PDF*.

> It contains **aggregated statistics, tables and charts only** — no individual records.


In [ ]:
_report_section('14. Generate the Results File to Send Back')
_doc = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>ELSA Vroegsignalering — Results</title>
<style>
 body{{font-family:-apple-system,Segoe UI,Arial,sans-serif;max-width:1000px;margin:24px auto;
       padding:0 18px;color:#222;line-height:1.4}}
 h1{{border-bottom:3px solid #2c7fb8;padding-bottom:6px}}
 h2.sec{{background:#2c7fb8;color:#fff;padding:6px 12px;border-radius:4px;margin-top:34px}}
 pre{{background:#f6f8fa;padding:8px 10px;border-radius:4px;white-space:pre-wrap;
      font-size:13px;font-family:SFMono-Regular,Consolas,monospace}}
 table{{border-collapse:collapse;margin:10px 0;font-size:13px}}
 th,td{{border:1px solid #ccc;padding:4px 9px;text-align:right}}
 th{{background:#eef3f7}}
 img{{max-width:100%;margin:12px 0;border:1px solid #eee;border-radius:4px}}
 .meta{{color:#666;font-size:13px}}
</style></head><body>
<h1>ELSA Vroegsignalering — Effectiveness Results</h1>
<p class="meta">Generated {_dt.datetime.now():%Y-%m-%d %H:%M} &middot; source file: {_html.escape(str(FILE))}
&middot; {len(_REPORT_BLOCKS)} content blocks</p>
<p class="meta"><em>Aggregated statistics, tables and charts only — no individual records.</em></p>
{''.join(_REPORT_BLOCKS)}
</body></html>'''

# Write next to the notebook's working directory and report the ABSOLUTE path
REPORT_PATH = Path(REPORT_PATH).resolve()
REPORT_PATH.write_text(_doc, encoding='utf-8')

_orig_print("=" * 70)
_orig_print("RESULTS FILE CREATED — this is the file to send back:")
_orig_print(f"   {REPORT_PATH}")
_orig_print("=" * 70)
_orig_print(f"  {len(_REPORT_BLOCKS)} content blocks captured (text, tables, charts).")
_orig_print(f"  Working directory: {Path.cwd()}")
_orig_print("  Open it in any browser; use Print -> Save as PDF if a PDF is preferred.")

# Clickable download link in Jupyter / JupyterLab
try:
    from IPython.display import FileLink
    _ipy_display(FileLink(str(REPORT_PATH)))
except Exception:
    pass

# In Google Colab, trigger a direct browser download automatically
try:
    from google.colab import files as _colab_files   # noqa
    _colab_files.download(str(REPORT_PATH))
except Exception:
    pass
